In [1]:
%run ./dataset.ipynb
%run ./model.ipynb

print("Training DataFrame:", train_df.shape)
print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Metadata path: E:\Shortcutscope\archive\waterbird_complete95_forest2water2\metadata.csv
Dataset shape: (11788, 6)
Columns: ['img_id', 'img_filename', 'y', 'split', 'place', 'place_filename']

First five rows:
   img_id                                                   img_filename  y  split  place                 place_filename
0       1  001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg  1      2      1          /o/ocean/00002178.jpg
1       2  001.Black_footed_Albatross/Black_Footed_Albatross_0009_34.jpg  1      0      1   /l/lake/natural/00000065.jpg
2       3  001.Black_footed_Albatross/Black_Footed_Albatross_0002_55.jpg  1      2      0  /b/bamboo_forest/00000131.jpg
3       4  001.Black_footed_Albatross/Black_Footed_Albatross_0074_59.jpg  1      0      1          /o/ocean/00001268.jpg
4       5  001.Black_footed_Albatross/Black_Footed_Albatross_0014_89.jpg  1      0      1          /o/ocean/00003147.jpg

Split column data type:
int64

Actual split values:
split
2    5

In [2]:
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preserve the configuration in the code you supplied
EPOCHS = 45
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 10
MIN_DELTA = 0.001
BATCH_SIZE = 32

MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = MODEL_DIR / "balanced_sampling_cnn.pt"
HISTORY_PATH = RESULTS_DIR / "balanced_sampling_history.csv"
TEST_METRICS_PATH = RESULTS_DIR / "balanced_sampling_test_metrics.json"

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)

Device: cpu
Checkpoint: models\balanced_sampling_cnn.pt


In [3]:
train_groups = (
    2 * train_df["y"].astype(int)
    + train_df["place"].astype(int)
).to_numpy()

if len(train_groups) != len(train_dataset):
    raise ValueError(
        f"train_df has {len(train_groups)} rows but train_dataset has "
        f"{len(train_dataset)} examples. Check that they use the same training split."
    )

group_counts = np.bincount(train_groups, minlength=4)

if np.any(group_counts == 0):
    raise ValueError(f"At least one training group is empty: {group_counts}")

group_sampling_weights = 1.0 / group_counts
example_sampling_weights = group_sampling_weights[train_groups]

balanced_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(example_sampling_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True
)

balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=0
)

# Show expected group sampling probabilities (approximately 25% each)
group_mass = np.array([
    example_sampling_weights[train_groups == g].sum()
    for g in range(4)
])
group_probabilities = group_mass / group_mass.sum()

print("Original training counts:", group_counts)
print("Expected group sampling probabilities:")
for group_id, probability in enumerate(group_probabilities):
    print(f"Group {group_id}: {probability:.3f}")

print("Balanced sampler is ready.")

Original training counts: [3498  184   56 1057]
Expected group sampling probabilities:
Group 0: 0.250
Group 1: 0.250
Group 2: 0.250
Group 3: 0.250
Balanced sampler is ready.


In [4]:
from sklearn.metrics import f1_score
def group_accuracies(predictions, labels, groups):
    result = {}
    for group_id in range(4):
        mask = groups == group_id
        if mask.sum().item() == 0:
            result[group_id] = float("nan")
        else:
            result[group_id] = (
                (predictions[mask] == labels[mask]).float().mean().item()
            )
    return result


def run_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []
    all_groups = []
    with torch.no_grad():
        for images, labels, groups in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            groups = groups.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size
            all_predictions.append(logits.argmax(dim=1).cpu())
            all_labels.append(labels.cpu())
            all_groups.append(groups.cpu())
    if total_examples == 0:
        raise ValueError("The DataLoader contains no examples.")
    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    groups = torch.cat(all_groups)
    # 1. Group-wise accuracy
    group_acc = group_accuracies(predictions, labels, groups)
    valid_acc = [acc for acc in group_acc.values() if not np.isnan(acc)]
    # 2. Overall Accuracy
    accuracy = (predictions == labels).float().mean().item()
    # 3. Macro F1 Score
    macro_f1 = f1_score(labels.numpy(), predictions.numpy(), average="macro")
    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "worst_group_accuracy": min(valid_acc),
        "group_accuracy": group_acc,
        "predictions": predictions,
        "labels": labels,
        "groups": groups
    }


In [5]:
torch.manual_seed(SEED)
model = ShortcutScopeCNN(num_classes=2).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-5
)
criterion = nn.CrossEntropyLoss()

best_score = -float("inf")
best_epoch = 0
best_state = None
no_improvement = 0
history = []

print("\\nTraining mode: GROUP-BALANCED SAMPLING")
print("Loss: Standard cross-entropy")
print("Group counts:", group_counts)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels, groups in balanced_train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)  # ordinary CE, no group-loss weights
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    if total_examples == 0:
        raise ValueError("The balanced training DataLoader is empty.")

    train_loss = total_loss / total_examples
    train_accuracy = total_correct / total_examples
    val = run_epoch(model, val_loader, criterion)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val["loss"],
        "val_accuracy": val["accuracy"],
        "val_worst_group_accuracy": val["worst_group_accuracy"],
        "learning_rate": optimizer.param_groups[0]["lr"]
    }
    for group_id, accuracy in val["group_accuracy"].items():
        row[f"val_group_{group_id}_accuracy"] = accuracy
    history.append(row)

    score = val["worst_group_accuracy"]
    if score > best_score + MIN_DELTA:
        best_score = score
        best_epoch = epoch + 1
        no_improvement = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, CHECKPOINT_PATH)
        print(
            f"  Best checkpoint saved at epoch {best_epoch} "
            f"(worst-group validation accuracy: {best_score:.4f})"
        )
    else:
        no_improvement += 1

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f} | "
        f"Val acc: {val['accuracy']:.4f} | "
        f"Worst-group val acc: {score:.4f} | "
        f"No improvement: {no_improvement}/{PATIENCE}"
    )

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    scheduler.step()

    if no_improvement >= PATIENCE:
        print(f"\\nEarly stopping at epoch {epoch + 1}.")
        break

if best_state is None:
    raise RuntimeError("No best checkpoint was saved.")

model.load_state_dict(best_state)
model.eval()

balanced_history = pd.DataFrame(history)
print("\\nTraining completed.")
print("Best epoch:", best_epoch)
print(f"Best worst-group validation accuracy: {best_score:.4f}")
print("Model saved to:", CHECKPOINT_PATH)
print("History saved to:", HISTORY_PATH)
display(balanced_history.tail())

\nTraining mode: GROUP-BALANCED SAMPLING
Loss: Standard cross-entropy
Group counts: [3498  184   56 1057]
  Best checkpoint saved at epoch 1 (worst-group validation accuracy: 0.1203)
Epoch 01/45 | Train loss: 0.7042 | Train acc: 0.5301 | Val acc: 0.7239 | Worst-group val acc: 0.1203 | No improvement: 0/10
  Best checkpoint saved at epoch 2 (worst-group validation accuracy: 0.4436)
Epoch 02/45 | Train loss: 0.6867 | Train acc: 0.5566 | Val acc: 0.5705 | Worst-group val acc: 0.4436 | No improvement: 0/10
Epoch 03/45 | Train loss: 0.6812 | Train acc: 0.5610 | Val acc: 0.6514 | Worst-group val acc: 0.2030 | No improvement: 1/10
Epoch 04/45 | Train loss: 0.6813 | Train acc: 0.5679 | Val acc: 0.6739 | Worst-group val acc: 0.2556 | No improvement: 2/10
Epoch 05/45 | Train loss: 0.6748 | Train acc: 0.5689 | Val acc: 0.6289 | Worst-group val acc: 0.2632 | No improvement: 3/10
Epoch 06/45 | Train loss: 0.6659 | Train acc: 0.5867 | Val acc: 0.6355 | Worst-group val acc: 0.2406 | No improvement: 4

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_worst_group_accuracy,learning_rate,val_group_0_accuracy,val_group_1_accuracy,val_group_2_accuracy,val_group_3_accuracy
17,18,0.644806,0.619187,0.712321,0.553795,0.437768,0.000141,0.653105,0.437768,0.481203,0.684211
18,19,0.639826,0.620647,0.622871,0.678899,0.225564,0.000134,0.895075,0.635193,0.225564,0.526316
19,20,0.628204,0.638165,0.684123,0.608841,0.360902,0.000128,0.837259,0.446352,0.360902,0.624060
20,21,0.634127,0.629614,0.624659,0.663053,0.293233,0.000121,0.832976,0.641631,0.293233,0.511278
21,22,0.619149,0.656517,0.613458,0.678899,0.248120,0.000115,0.884368,0.658798,0.248120,0.458647


In [6]:
best_model = ShortcutScopeCNN(num_classes=2).to(DEVICE)
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=True
)
best_model.load_state_dict(checkpoint)

test_result = run_epoch(
    best_model,
    test_loader,
    nn.CrossEntropyLoss()
)

print("\\nGROUP-BALANCED SAMPLING — TEST RESULTS")
print(f"Test accuracy: {test_result['accuracy']:.4f}")
print(f"Worst-group test accuracy: {test_result['worst_group_accuracy']:.4f}")
for group_id, accuracy in test_result["group_accuracy"].items():
    print(f"Group {group_id} accuracy: {accuracy:.4f}")

test_metrics = {
    "experiment": "group_balanced_sampling",
    "best_epoch": int(best_epoch),
    "best_worst_group_validation_accuracy": float(best_score),
    "test_accuracy": float(test_result["accuracy"]),
    "worst_group_test_accuracy": float(test_result["worst_group_accuracy"]),
    "group_accuracy": {
        str(group_id): float(accuracy)
        for group_id, accuracy in test_result["group_accuracy"].items()
    }
}

with open(TEST_METRICS_PATH, "w") as file:
    json.dump(test_metrics, file, indent=4)

print("Test metrics saved to:", TEST_METRICS_PATH)

\nGROUP-BALANCED SAMPLING — TEST RESULTS
Test accuracy: 0.5537
Worst-group test accuracy: 0.4754
Group 0 accuracy: 0.5867
Group 1 accuracy: 0.4754
Group 2 accuracy: 0.5826
Group 3 accuracy: 0.6838
Test metrics saved to: results\balanced_sampling_test_metrics.json


In [2]:
import shutil
import json
import hashlib
from pathlib import Path

# ----------------------------------------------------
# 1. Setup Frozen Archive Directory
# ----------------------------------------------------
FROZEN_DIR = Path("frozen_experiment_balanced_sampling")
FROZEN_DIR.mkdir(parents=True, exist_ok=True)
(FROZEN_DIR / "models").mkdir(exist_ok=True)
(FROZEN_DIR / "results").mkdir(exist_ok=True)

# ----------------------------------------------------
# 2. Archive Model Checkpoint & Training Results
# ----------------------------------------------------
files_to_archive = {
    Path("models/balanced_sampling_cnn.pt"): FROZEN_DIR / "models/balanced_sampling_cnn.pt",
    Path("results/balanced_sampling_history.csv"): FROZEN_DIR / "results/balanced_sampling_history.csv",
    Path("results/balanced_sampling_test_metrics.json"): FROZEN_DIR / "results/balanced_sampling_test_metrics.json",
}

for src, dst in files_to_archive.items():
    if not src.exists():
        raise FileNotFoundError(f"Missing file to archive: {src.resolve()}")
    shutil.copy2(src, dst)
    print(f"Archived: {src} -> {dst}")

# ----------------------------------------------------
# 3. Save Experiment Configuration & Transforms
# ----------------------------------------------------
config = {
    "project": "ShortcutScope",
    "experiment": "Group-Balanced Sampling Mitigation",
    "dataset": "Waterbirds (95% Spurious Correlation)",
    "architecture": "ShortcutScopeCNN (4-Block ConvNet with BatchNorm & ReLU)",
    "input_size": [128, 128],
    "batch_size": 32,
    "sampling_method": "WeightedRandomSampler (Inverse Group Frequency - 25% per group)",
    "loss_function": "Standard Cross-Entropy (unweighted loss)",
    "optimizer": "AdamW",
    "learning_rate": 2e-4,
    "weight_decay": 1e-2,
    "dropout": 0.3,
    "gradient_clipping": "max_norm=1.0",
    "scheduler": "CosineAnnealingLR (eta_min=1e-5)",
    "early_stopping_patience": 10,
    "checkpoint_selection": "Best validation worst-group accuracy",
    "train_augmentations": [
        "RandomResizedCrop(128, scale=(0.7, 1.0))",
        "RandomHorizontalFlip(p=0.5)",
        "ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)"
    ],
    "eval_transforms": [
        "Resize(128, 128)",
        "Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])"
    ]
}

with open(FROZEN_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

print("\nConfiguration and experiment artifacts archived successfully.")

# ----------------------------------------------------
# 4. Generate SHA-256 Manifest
# ----------------------------------------------------
def sha256_file(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

manifest = {}
for file_path in sorted(FROZEN_DIR.rglob("*")):
    if file_path.is_file() and file_path.name != "manifest.json":
        manifest[str(file_path.relative_to(FROZEN_DIR))] = {
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256_file(file_path)
        }

with open(FROZEN_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("=" * 65)
print(f"Frozen experiment location: {FROZEN_DIR.resolve()}")
print(f"Files archived: {len(manifest)}")
print("Manifest with SHA-256 hashes saved successfully.")
print("=" * 65)


Archived: models\balanced_sampling_cnn.pt -> frozen_experiment_balanced_sampling\models\balanced_sampling_cnn.pt
Archived: results\balanced_sampling_history.csv -> frozen_experiment_balanced_sampling\results\balanced_sampling_history.csv
Archived: results\balanced_sampling_test_metrics.json -> frozen_experiment_balanced_sampling\results\balanced_sampling_test_metrics.json

Configuration and experiment artifacts archived successfully.
Frozen experiment location: E:\Shortcutscope\frozen_experiment_balanced_sampling
Files archived: 4
Manifest with SHA-256 hashes saved successfully.
